# Which suppliers to contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/supplier_selection/notebooks/supplier_selection.ipynb)

A company assembles 5,000 electric bicycles a year and buys six components: frames, motors, batteries, wheels, brakes and displays. It decides which suppliers to contract and which supplier each component is bought from. Each contracted supplier costs a fixed amount a year, besides what is bought from it. The notebook follows the decision in steps:

1. A weighted decision matrix for one component, with and without the price.
2. The rule a buyer would apply: for each component, the cheapest acceptable supplier.
3. The number of possible selections, and the optimal selection given by an integer model.
4. A local search that starts from a selection and improves it one move at a time.
5. A reduced version that shows how a solver handles an integer model: linear relaxation, rounding, and branch and bound.
6. A decision tree that compares the buyer's rule with buying everything from the distributor when a supplier may stop delivering during the year.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and tick boxes): change the values and run the cell again. Outside Colab the form is a line such as `A = True`: edit the value and run the cell.

All the data and results are explained in the [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/supplier_selection/supplier_selection.md). The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/supplier_selection"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    if "suppliers" in sys.modules:
        print("The example was already loaded in this session: restart it "
              "(Runtime > Restart session) and run this cell again.")
import suppliers as ss

## 1. Prices, fixed costs and supplier scorecard

A, B, C and D are specialised manufacturers; E is a distributor that offers all six components; F offers only batteries. The first table gives the price of each component at each supplier, and a dash means that the supplier does not offer it. The second gives the annual cost of buying the whole volume of a component from one supplier, in thousands of euros a year (k€/year): price × 5,000 / 1,000. Each contracted supplier also costs 50 k€/year (qualification, audits, quality engineering and contract management).

The scorecard rates each supplier from 0 to 10 on four criteria. Its weighted score is the sum of the four scores times their weights. A supplier is acceptable if its weighted score is at least 6: F is not.

In [ ]:
# @title Data { display-mode: "form" }
ss.show_data()

## 2. Weighted decision matrix for the batteries

Four suppliers offer batteries: B, C, E and F. Without the price, the matrix ranks them by the weighted score of the scorecard. With the price as one more criterion, each supplier gets a price score from 0 to 10: 10 for the cheapest, 0 for the dearest, and in proportion between them, $10 \times (p_{\max} - p)/(p_{\max} - p_{\min})$. The price gets the weight you choose, and the four criteria of the scorecard share the rest in their usual proportions.

The slider sets the weight of the price, from 0 % to 60 %. The tick box leaves the offer of F out of the comparison: the highest and lowest prices are then taken among B, C and E. The table gives each supplier's scores and its weighted score without and with the price, from the highest; the bar chart compares the two scores with the line of an acceptable supplier.

At 40 %, F, whose quality score is 4, comes second. From which weight of the price does F rank above B? What does that say about adding the price to the same matrix as quality? Untick F: which scores change, if nothing has changed in B, C or E?

In [ ]:
# @title Decision matrix for the batteries { display-mode: "form" }
price_weight = 40  # @param {type:"slider", min:0, max:60, step:5}
include_F = True  # @param {type:"boolean"}
ss.batteries_matrix(price_weight, include_F)

The company therefore uses the scorecard only to decide which suppliers are acceptable (score of at least 6), and decides in euros among the acceptable ones.

## 3. The buyer's rule

A buyer would take each component on its own and buy it from the cheapest acceptable supplier. The cell gives the supplier and the annual cost of each component, the suppliers contracted and the total: purchases plus fixed costs. It also gives the cost of buying everything from the distributor E. E is the cheapest supplier of no component, so the rule never chooses it. Why does E alone cost less than the rule?

In [ ]:
# @title The buyer's rule { display-mode: "form" }
ss.buyer_rule()

### Cost of any selection of suppliers

A selection is the set of suppliers that the company contracts. Given the set, each component is bought from the cheapest contracted supplier that offers it. Tick the suppliers to contract and run the cell. If some component has no contracted supplier, the cell says which one. Otherwise it gives the purchases, the fixed costs and the total, compared with the buyer's rule, and a bar with the two parts of the cost.

The form starts from the selection of the rule. Can you find a selection that costs less than the rule? How many suppliers does it contract?

In [ ]:
# @title Cost of a selection { display-mode: "form" }
A = True  # @param {type:"boolean"}
B = True  # @param {type:"boolean"}
C = True  # @param {type:"boolean"}
D = True  # @param {type:"boolean"}
E = False  # @param {type:"boolean"}
ss.check_selection([s for s, chosen in zip("ABCDE", [A, B, C, D, E]) if chosen])

## 4. Number of selections and optimal selection

With five acceptable suppliers there are $2^5 - 1 = 31$ non-empty sets of suppliers; only some of them cover the six components. The cell counts them, shows how $2^n - 1$ grows with the number of suppliers $n$, and lists the selections that cover every component, from the cheapest, with their purchases and fixed costs. The last column marks the local optima of the search of section 5.

How far is the selection of the buyer's rule from the top of the list?

In [ ]:
# @title All the selections { display-mode: "form" }
ss.count_selections()

### Integer model

$S$ is the set of acceptable suppliers and $M$ the set of components. $c_{sm}$ is the annual cost of buying component $m$ from supplier $s$ (k€/year), $f_s$ the fixed annual cost of a contracted supplier (k€/year) and $a_{sm}$ is 1 if $s$ makes an acceptable offer for $m$ and 0 otherwise. The decisions are binary: $y_s = 1$ if $s$ is contracted, and $x_{sm} = 1$ if $m$ is bought from $s$.

$$
\begin{aligned}
\min \quad & \sum_{s \in S} f_s\, y_s + \sum_{s \in S} \sum_{m \in M} c_{sm}\, x_{sm} && && \text{(annual cost)} \\
\text{s.t.} \quad & \sum_{s \in S} x_{sm} = 1 && \forall m \in M && \text{(one supplier per component)} \\
& x_{sm} \le a_{sm} && \forall s \in S,\ m \in M && \text{(only an acceptable offer)} \\
& x_{sm} \le y_s && \forall s \in S,\ m \in M && \text{(bought from implies contracted)} \\
& x_{sm},\ y_s \in \{0,1\} && \forall s \in S,\ m \in M
\end{aligned}
$$

The cell writes the model with the library PuLP and the solver HiGHS solves it. It gives the optimal selection and compares its cost with the rule, purchases and fixed costs side by side. Where does the model save money: in the purchases or in the fixed costs?

In [ ]:
# @title Optimal selection of the model { display-mode: "form" }
ss.solve_model()

## 5. Local search

A local search starts from a selection and changes it one move at a time. A move removes a supplier, adds one, or swaps a contracted supplier for one that is not contracted; a move that leaves a component without a supplier is not allowed. At each step the search makes the move that lowers the cost most, and it stops when no move lowers it: the selection where it stops is a local optimum.

Tick the suppliers of the starting selection. The cell prints each step with its cost, the neighbours of the selection where the search stops, from the cheapest, and the moves that are not allowed; the chart shows the cost step by step against the optimum of the model.

The form starts from the selection of the buyer's rule. From which starting selections does the search reach the optimum of the model? From $\{B, C, D\}$, which two moves lead to it, and what happens to the cost after the first one?

In [ ]:
# @title Local search { display-mode: "form" }
A = True  # @param {type:"boolean"}
B = True  # @param {type:"boolean"}
C = True  # @param {type:"boolean"}
D = True  # @param {type:"boolean"}
E = False  # @param {type:"boolean"}
ss.local_search_steps([s for s, chosen in zip("ABCDE", [A, B, C, D, E]) if chosen])

## 6. Reduced version: how an integer model is solved

Three suppliers and three components, with the same costs: A offers frames and motors, B offers motors and batteries, C offers frames and batteries. No supplier covers the three components on its own. The table gives the annual cost of each selection.

In [ ]:
# @title Reduced version { display-mode: "form" }
ss.reduced_data()

### Linear relaxation and rounding

The linear relaxation is the same model with every variable allowed to take any value between 0 and 1. It is a linear model, easier to solve, and no selection can cost less than its optimum: that optimum is a **bound**. The cell gives the values of $y$ and $x$ in the relaxation, the bound and the two obvious roundings of $y$. Is rounding a good way to get the optimal selection?

In [ ]:
# @title Linear relaxation { display-mode: "form" }
ss.relaxation()

### Branch and bound

Branch and bound splits the problem in two by fixing one fractional variable to 1 in one branch and to 0 in the other, and solves the relaxation of each part. The rules used here:

- branch on the $y_s$ closest to 0.5 (alphabetical order if two are equally close);
- the branch $y_s = 1$ first, then $y_s = 0$; one branch to the end before the other;
- a node is closed if its relaxation has no solution, if its solution is integer (it becomes the best known if it is cheaper), or if its bound is not better than the best known.

"Bought from implies contracted" can also be written with one constraint per supplier, $\sum_{m} x_{sm} \le n_s\, y_s$, where $n_s$ is the number of components that $s$ offers (2 for each supplier here). Both ways give the same selections; their relaxations differ. The cell solves the reduced version with each one and gives, for each node, the variables fixed, the bound, the values of $y$ and what happens to it, with a drawing of the tree.

Which way of writing the model gives the higher bound at the first node, and how many nodes does each tree need?

In [ ]:
# @title Two branch-and-bound trees { display-mode: "form" }
ss.branch_and_bound()

## 7. Decision tree with a supplier that may stop delivering

The model counts only costs. Buying everything from E costs less than the buyer's rule, but the company then depends on one supplier. The decision tree compares the two strategies when a supplier may stop delivering during the year. In each strategy, the supplier that serves more than one component is uncertain: E, which serves the six components, or C, which serves frames, batteries and wheels in the rule. The suppliers that serve one component are treated as reliable. When the uncertain supplier stops, each of its components is moved to another supplier at short notice, at a cost per component: lost assembly, urgent transport and qualification of the new supplier.

The form sets the probability that E stops, the probability that C stops and the cost of moving one component. The first table gives each branch of the tree with its probability and its annual cost. The second gives, for each strategy, the planned cost, the expected cost (the cost of each branch times its probability, added up) and the worst case. The cell then says which strategy has the lower expected cost and which the lower worst case, and below which probability that E stops buying everything from E has the lower expected cost. The drawing shows the tree.

In a given year only one branch happens. If the company could not afford a year like the worst case of buying everything from E, which strategy would it choose, and does the expected cost give the same answer?

In [ ]:
# @title Decision tree { display-mode: "form" }
stop_E = 0.05  # @param {type:"slider", min:0, max:0.3, step:0.01}
stop_C = 0.10  # @param {type:"slider", min:0, max:0.3, step:0.01}
move_cost = 100  # @param {type:"slider", min:0, max:300, step:10}
ss.disruption_tree(stop_E, stop_C, move_cost)

All the numbers are explained in the [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/supplier_selection/supplier_selection.md), which also has four scripts that reproduce them.